# Primes and $\ln$: the Prime Number Theorem

This notebook finds the natural logarithm somewhere nobody would expect it: in the distribution of the prime numbers. Primes look erratic up close, but on a large scale they thin out in a perfectly regular way — near $n$, roughly one number in $\ln n$ is prime. The **prime number theorem** makes this precise: $\pi(n) \sim n/\ln n$.

This is notebook 05 of module 04 (**Logarithms in calculus**) in the **Learning Logarithms** series.

## 1. Introduction

**How many prime numbers are there below one billion?**

Checking each of the $10^9$ numbers for divisors would take a long time even for a computer, and by hand it is hopeless. Yet there is a one-line estimate that gets within about 5%:

$$
\pi(10^9) \approx \frac{10^9}{\ln 10^9} = \frac{10^9}{20.72} \approx 48{,}255{,}000 ,
$$

and a slightly better one that gets within $0.004\%$:

$$
\pi(10^9) \approx \operatorname{Li}(10^9) = \int_2^{10^9} \frac{dt}{\ln t} \approx 50{,}849{,}234 .
$$

The true count is $\pi(10^9) = 50{,}847{,}534$ — and in this notebook we will count them ourselves, in about two seconds, with a NumPy sieve.

Here $\pi(n)$ denotes the number of primes $\le n$ (nothing to do with $3.14159\ldots$). Why should a **logarithm** describe primes, which are defined by divisibility and seem to have nothing to do with continuous functions? The idea goes back to tables of primes. In a letter written in 1849 to the astronomer Johann Encke, Carl Friedrich Gauss recalled that as a teenager, around 1792–93, he had counted primes in blocks of numbers and noticed that their **density** near $n$ is about $1/\ln n$. Adding up a density gives an integral — which is why $\operatorname{Li}(n)$ appears. Adrien-Marie Legendre published a similar empirical formula, $n/(\ln n - 1.08366)$, in 1798 and 1808. A proof took a century: the theorem was proved in 1896, independently by Jacques Hadamard and Charles de la Vallée Poussin, building on Bernhard Riemann's 1859 work on the zeta function.

**Prerequisites**

- Growth rates and $\sim$ notation: $\ln n \ll n$ — [Growth rates and asymptotics](../02_graphs_equations_and_scales/04_growth_rates_and_asymptotics.ipynb) (this series).
- Log-scaled axes — [Log scales and power laws](../02_graphs_equations_and_scales/03_log_scales_and_power_laws.ipynb) (this series).
- Integrals as accumulated density, and numerical integration — [Introduction to integrals](../../../LearningCalculus/hypersklill/04_integrals/00_introduction_to_integrals.ipynb) (LearningCalculus).
- Substitution in integrals, used to compute $\operatorname{Li}$ — [U-substitution](../../../LearningCalculus/hypersklill/04_integrals/03_u_substitution.ipynb) (LearningCalculus).
- Sums of logarithms and their integral approximations — [Stirling's approximation](04_stirlings_approximation.ipynb) (previous notebook).

After completing this notebook you will be able to:

- State the prime number theorem in its two common forms, $\pi(n) \sim n/\ln n$ and $\pi(n) \sim \operatorname{Li}(n)$, and explain the difference between them.
- Implement the sieve of Eratosthenes in NumPy and count primes up to $10^9$.
- Compute $\operatorname{Li}(x)$ by numerical integration using only NumPy.
- Interpret $1/\ln n$ as the "probability that a number near $n$ is prime" and $\ln n$ as the average gap between primes near $n$.
- Estimate prime counts and gaps at scales far beyond any computer, such as 100-digit numbers.

## 2. Theory

### 2.1 The prime-counting function

$$
\pi(n) = \#\{\,p \le n : p \text{ prime}\,\}, \qquad \pi(10) = 4 \;(2, 3, 5, 7), \quad \pi(100) = 25 .
$$

$\pi$ is a staircase that steps up by $1$ at each prime. Euclid showed there are infinitely many primes, so $\pi(n) \to \infty$; the question is **how fast**.

### 2.2 Density $1/\ln n$

Count primes in blocks of $1000$ consecutive numbers: there are $168$ in $[1, 1000]$, but only about $72$ near $10^6$ and about $54$ near $10^8$ (on average). The fraction of primes near $n$ — the **density** — keeps falling, and Gauss's observation is that it falls like

$$
\text{density of primes near } n \approx \frac{1}{\ln n} .
$$

The logarithm turns the multiplicative growth of $n$ into additive growth of the gap: each time $n$ is multiplied by $10$, the typical spacing between primes grows by the **same amount**, $\ln 10 \approx 2.3$. Near $10^3$ about one number in $7$ is prime; near $10^6$, one in $14$; near $10^9$, one in $21$; near $10^{12}$, one in $28$.

A heuristic reason: a random number near $n$ is prime when it has no prime factor up to $\sqrt n$, and the product $\prod_{p \le \sqrt n}(1 - 1/p)$ shrinks like a constant over $\ln n$ (a result of Mertens). The heuristic gets the shape right but not the constant; the real proof is much harder.

### 2.3 Adding up the density: $\operatorname{Li}(n)$ and $n/\ln n$

If the density near $t$ is $1/\ln t$, the number of primes up to $n$ is the accumulated density:

$$
\pi(n) \approx \operatorname{Li}(n) = \int_2^n \frac{dt}{\ln t} .
$$

This integral has no elementary antiderivative. Integrating by parts once ($u = 1/\ln t$, $dv = dt$):

$$
\operatorname{Li}(n) = \frac{n}{\ln n} - \frac{2}{\ln 2} + \int_2^n \frac{dt}{(\ln t)^2} = \frac{n}{\ln n} + \frac{n}{(\ln n)^2} + \frac{2n}{(\ln n)^3} + \cdots
$$

so the simplest approximation is $n/\ln n$, and $\operatorname{Li}(n)$ is $n/\ln n$ plus corrections that are smaller by factors of $\ln n$. Since $\ln n$ grows so slowly, those corrections matter a lot in practice: at $n = 10^9$, $\frac{n}{(\ln n)^2}$ is still about $5\%$ of $\frac{n}{\ln n}$.

### 2.4 The prime number theorem

$$
\pi(n) \sim \frac{n}{\ln n} \qquad\text{and}\qquad \pi(n) \sim \operatorname{Li}(n), \qquad\text{meaning}\quad \lim_{n\to\infty}\frac{\pi(n)}{n/\ln n} = \lim_{n\to\infty}\frac{\pi(n)}{\operatorname{Li}(n)} = 1 .
$$

Both statements are equivalent (because $\operatorname{Li}(n) \sim n/\ln n$), but $\operatorname{Li}$ is far more accurate: the ratio $\pi(n)/(n/\ln n)$ creeps toward $1$ from above at a painfully slow rate (it is still $1.054$ at $10^9$), while $\pi(n)/\operatorname{Li}(n)$ is already $0.99997$ there.

The theorem says nothing exact about any single prime; it is a statement about averages, like a law of large numbers.

### 2.5 Gaps between primes

If one number in $\ln n$ is prime near $n$, the **average gap** between consecutive primes near $n$ is about $\ln n$. Individual gaps vary wildly — twin primes have gap $2$, and there are long prime-free stretches — but their average is $\ln n$.

In log form: near $n = 10^k$, the average gap is $\ln 10^k = k\ln 10 \approx 2.3\,k$. Gaps grow **linearly in the number of digits**.

### 2.6 Where the theorem is used

Cryptography needs large random primes (hundreds of digits). The prime number theorem says that a random $d$-digit number is prime with probability about $1/(d\ln 10)$ — for $d = 300$, about $1$ in $690$, or $1$ in $345$ among odd numbers. So a computer finds such a prime by testing a few hundred random odd candidates: primes are rare, but not too rare.

## 3. Worked Examples

### Example 1 — primes below a billion

$\ln 10^9 = 9\ln 10 \approx 9 \times 2.302585 = 20.723$. So $\dfrac{10^9}{\ln 10^9} \approx 48{,}254{,}942$. The true value $50{,}847{,}534$ is $5.4\%$ larger. $\operatorname{Li}(10^9) \approx 50{,}849{,}234$ is too large by only $1{,}700$.

### Example 2 — is a random number near a million prime?

$\ln 10^6 \approx 13.8$: about one number in $14$ is prime. Among odd numbers, one in $7$. Indeed there are $\pi(10^6) - \pi(999{,}000) = 65$ primes in the last thousand numbers below $10^6$; the density predicts $1000/13.8 \approx 72$. A single block of a thousand fluctuates by about $\pm 8$ around the prediction; the density describes the average.

### Example 3 — average gap

Near $10^8$: $\ln 10^8 = 18.42$, so consecutive primes near a hundred million are on average about $18$ apart. Near $10^{100}$: about $230$ apart.

### Example 4 — the first terms of $\operatorname{Li}$ at $n = 10^6$

$\dfrac{n}{\ln n} = \dfrac{10^6}{13.8155} = 72{,}382$; adding $\dfrac{n}{(\ln n)^2} = 5{,}239$ gives $77{,}621$; adding $\dfrac{2n}{(\ln n)^3} = 758$ gives $78{,}380$. The true $\pi(10^6) = 78{,}498$ and $\operatorname{Li}(10^6) \approx 78{,}627$.

## 4. Python Examples

### 4.1 The sieve of Eratosthenes in NumPy

Start with "every number is prime"; for each $p$ up to $\sqrt{n}$ that is still marked prime, cross out its multiples $p^2, p^2 + p, \dots$ (smaller multiples were already crossed out by smaller primes). In NumPy, crossing out is a single slice assignment `is_prime[p*p::p] = False`. The functions defined in this cell are used by later cells.

In [ ]:
import math

import numpy as np


def sieve(limit):
    # Boolean array: is_prime[k] is True exactly when k is prime (0 <= k <= limit).
    is_prime = np.ones(limit + 1, dtype=bool)
    is_prime[:2] = False
    for p in range(2, math.isqrt(limit) + 1):
        if is_prime[p]:
            is_prime[p * p :: p] = False
    return is_prime


def primes_up_to(limit):
    return np.flatnonzero(sieve(limit))


def prime_count(n, primes):
    # pi(n) for every n in the array, using a sorted array of primes
    return np.searchsorted(primes, n, side="right")


print("primes below 50:", primes_up_to(50))
primes = primes_up_to(10**8)
print("number of primes up to 10^8:", len(primes), "  largest:", primes[-1])
for exponent in range(1, 9):
    print(f"pi(10^{exponent}) = {prime_count(10**exponent, primes):>10,}")

### 4.2 Counting to a billion: a segmented sieve

A boolean array with $10^9$ entries needs a gigabyte of memory. A **segmented** sieve processes the range in blocks of ten million: first find the primes up to $\sqrt{10^9} \approx 31{,}623$, then cross out their multiples block by block, counting what survives.

In [ ]:
import math
import time

import numpy as np


def count_primes_segmented(limit, block_size=10**7):
    small_primes = primes_up_to(math.isqrt(limit))
    count = 0
    for block_start in range(0, limit + 1, block_size):
        block_end = min(block_start + block_size, limit + 1)
        block = np.ones(block_end - block_start, dtype=bool)
        if block_start == 0:
            block[:2] = False
        for p in small_primes:
            first_multiple = max(p * p, ((block_start + p - 1) // p) * p)
            if first_multiple < block_end:
                block[first_multiple - block_start :: p] = False
        count += int(block.sum())
    return count


start_time = time.perf_counter()
pi_billion = count_primes_segmented(10**9)
elapsed = time.perf_counter() - start_time
print(f"pi(10^9) = {pi_billion:,}   (computed in {elapsed:.1f} s)")
print(f"10^9 / ln(10^9) = {10**9 / math.log(10**9):,.0f}")

### 4.3 $\operatorname{Li}(x)$ by numerical integration

The integrand $1/\ln t$ varies slowly but the range is huge, so a uniform grid in $t$ wastes points. Substitute $t = e^u$, $dt = e^u\,du$:

$$
\operatorname{Li}(x) = \int_2^x \frac{dt}{\ln t} = \int_{\ln 2}^{\ln x} \frac{e^u}{u}\,du ,
$$

an integral over a short interval ($[0.69, 20.7]$ for $x = 10^9$), which the trapezoid rule handles well with a couple of hundred thousand points. This is the logarithm doing what it always does: turning a multiplicative range into an additive one.

In [ ]:
import math

import numpy as np

if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz


def Li(x, points=200_001):
    u = np.linspace(math.log(2), math.log(x), points)
    return np.trapezoid(np.exp(u) / u, u)


# Check against a much finer grid
for x in [10**3, 10**6, 10**9]:
    print(f"Li({x:.0e}) = {Li(x):,.3f}   (with 10x more points: {Li(x, 2_000_001):,.3f})")

### 4.4 The table: $\pi(n)$, $n/\ln n$ and $\operatorname{Li}(n)$

In [ ]:
import math

print(f"{'n':>6} {'pi(n)':>12} {'n/ln n':>14} {'Li(n)':>14} {'pi/(n/ln n)':>12} {'pi/Li':>9} {'Li - pi':>9}")
for exponent in range(2, 10):
    n = 10**exponent
    true_count = pi_billion if exponent == 9 else int(prime_count(n, primes))
    simple = n / math.log(n)
    integral = Li(n)
    print(f"10^{exponent:<3} {true_count:>12,} {simple:>14,.0f} {integral:>14,.0f} "
          f"{true_count / simple:>12.4f} {true_count / integral:>9.5f} {integral - true_count:>9,.0f}")

The ratio $\pi(n)/(n/\ln n)$ falls only slowly toward $1$ ($1.13$ at $10^4$, $1.05$ at $10^9$), while $\pi(n)/\operatorname{Li}(n)$ is within $0.5\%$ of $1$ at $10^5$ and within $0.05\%$ at $10^7$. In this whole range $\operatorname{Li}(n)$ overestimates $\pi(n)$ by a relatively tiny amount, between one and two times $\sqrt n / \ln n$.

### 4.5 Average gap near $n$ is $\ln n$

Take all primes in a window just below $10^k$ and compute the mean gap between consecutive ones.

In [ ]:
import math

import numpy as np

for exponent in range(3, 9):
    n = 10**exponent
    window = n // 10
    in_window = primes[(primes > n - window) & (primes <= n)]
    gaps = np.diff(in_window)
    print(f"near 10^{exponent}: {len(in_window):>7,} primes, mean gap {gaps.mean():7.3f},  "
          f"ln(n) = {math.log(n):7.3f},  largest gap {gaps.max():>4}")

The mean gaps track $\ln n$ closely; the small windows near $10^3$ and $10^4$ contain few primes and fluctuate more. The largest gaps are several times the mean — individual gaps are irregular, only the average is predictable.

## 5. Visualizations

### 5.1 $\pi(n)$ up close and from far away

Left: up to $n = 200$, $\pi(n)$ is a staircase, and $n/\ln n$ and $\operatorname{Li}(n)$ are smooth curves on either side of it. Right: on log-log axes up to $10^8$ the three are nearly indistinguishable — the irregularity of the primes disappears on a large scale.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

small_n = np.arange(2, 201)
large_n = np.unique(np.logspace(1, 8, 200).astype(np.int64))

fig, (ax_small, ax_large) = plt.subplots(1, 2, figsize=(12, 4.8))
ax_small.step(small_n, prime_count(small_n, primes), where="post", color="black", label="$\\pi(n)$")
ax_small.plot(small_n, small_n / np.log(small_n), color="tab:orange", label="$n/\\ln n$")
ax_small.plot(small_n, [Li(n, 2001) for n in small_n], color="tab:blue", label="Li(n)")
ax_small.set_xlabel("n")
ax_small.set_ylabel("number of primes")
ax_small.grid(True, alpha=0.3)
ax_small.legend(loc="upper left")
ax_small.set_title("Up to 200: a staircase")

ax_large.plot(large_n, prime_count(large_n, primes), color="black", linewidth=2.5, label="$\\pi(n)$")
ax_large.plot(large_n, large_n / np.log(large_n), "--", color="tab:orange", label="$n/\\ln n$")
ax_large.plot(large_n, [Li(n, 20001) for n in large_n], ":", color="tab:blue", linewidth=2, label="Li(n)")
ax_large.set_xscale("log")
ax_large.set_yscale("log")
ax_large.set_xlabel("n (log scale)")
ax_large.set_ylabel("number of primes (log scale)")
ax_large.grid(True, which="both", alpha=0.3)
ax_large.legend(loc="upper left")
ax_large.set_title("Up to $10^8$: smooth")
plt.tight_layout()
plt.show()

### 5.2 The ratios approach 1 — at very different speeds

$\pi(n)/(n/\ln n)$ and $\pi(n)/\operatorname{Li}(n)$ against $n$ on a logarithmic $x$-axis. The theorem says both tend to $1$. The second hugs $1$ almost immediately; the first approaches it so slowly that, on a log axis, it barely looks like converging — the leftover is roughly $1 + 1/\ln n$, and $\ln n$ grows very slowly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_values = np.unique(np.logspace(2, 8, 150).astype(np.int64))
counts = prime_count(n_values, primes)
ratio_simple = counts / (n_values / np.log(n_values))
ratio_li = counts / np.array([Li(n, 20001) for n in n_values])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_values, ratio_simple, color="tab:orange", label="$\\pi(n) / (n/\\ln n)$")
ax.plot(n_values, 1 + 1 / np.log(n_values), "--", color="tab:orange", alpha=0.6, label="$1 + 1/\\ln n$")
ax.plot(n_values, ratio_li, color="tab:blue", linewidth=2, label="$\\pi(n)$ / Li(n)")
ax.axhline(1, color="black", linewidth=0.8)
ax.set_xscale("log")
ax.set_xlabel("n (log scale)")
ax.set_ylabel("ratio")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper right")
ax.set_title("Prime number theorem: both ratios tend to 1")
plt.show()

### 5.3 Density of primes against $1/\ln n$

The fraction of primes in consecutive blocks of $100{,}000$ numbers (dots) against $1/\ln n$ (line). On a logarithmic $x$-axis, $1/\ln n$ is a smooth slowly falling curve, and the measured densities sit right on it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

block = 100_000
block_starts = np.unique(np.logspace(np.log10(block), 8, 60).astype(np.int64) // block * block)
block_starts = block_starts[block_starts + block <= 10**8]
densities = (prime_count(block_starts + block, primes) - prime_count(block_starts, primes)) / block
centres = block_starts + block / 2

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(centres, densities, "o", color="tab:blue", label="fraction of primes in a block of 100,000")
smooth_n = np.logspace(np.log10(block), 8, 300)
ax.plot(smooth_n, 1 / np.log(smooth_n), color="tab:red", label="$1/\\ln n$")
ax.set_xscale("log")
ax.set_xlabel("n (log scale)")
ax.set_ylabel("density of primes")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper right")
ax.set_title("The chance that a number near n is prime is about $1/\\ln n$")
plt.show()

### 5.4 Gaps between primes near $10^8$

A histogram of the gaps between consecutive primes in the last million numbers below $10^8$. Gaps are always even (apart from $2 \to 3$), they vary from $2$ to well over $100$, and their mean is close to $\ln 10^8 \approx 18.4$. The roughly exponential shape on the log $y$-axis (a straight-ish decline) is what a "random" model with density $1/\ln n$ predicts.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

n = 10**8
in_window = primes[primes > n - 10**6]
gaps = np.diff(in_window)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(gaps, bins=np.arange(1, gaps.max() + 3, 2), color="tab:blue", alpha=0.7, edgecolor="white")
ax.axvline(gaps.mean(), color="black", linewidth=2, label=f"mean gap = {gaps.mean():.2f}")
ax.axvline(math.log(n), color="crimson", linestyle="--", linewidth=2, label=f"ln(10^8) = {math.log(n):.2f}")
ax.set_yscale("log")
ax.set_xlabel("gap between consecutive primes")
ax.set_ylabel("count (log scale)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper right")
ax.set_title("Prime gaps just below $10^8$")
plt.show()

## 6. Exercises

### Basic

1. **Estimate it first, then compute.** Using $\ln 10 \approx 2.3$, estimate $\pi(10^6)$ with $n/\ln n$. Compare with the true value $78{,}498$: by what percentage is the estimate off?
2. Estimate the probability that a randomly chosen integer near $500{,}000$ is prime. Check by counting the primes between $490{,}000$ and $510{,}000$.
3. Sieve by hand (or in your head) the primes below $50$. How many are there? Compare with $50/\ln 50$ and $\operatorname{Li}(50)$.

### Intermediate

4. Estimate the average gap between consecutive primes near $10^{12}$ and near $10^{100}$.
5. Roughly how many primes have exactly $100$ digits? What fraction of $100$-digit numbers is prime? What fraction of **odd** $100$-digit numbers?
6. Compare Legendre's formula $\dfrac{n}{\ln n - 1.08366}$, the variant $\dfrac{n}{\ln n - 1}$ and $\operatorname{Li}(n)$ with $\pi(10^7) = 664{,}579$ and $\pi(10^9) = 50{,}847{,}534$. Which is best at each?

### Advanced

7. Derive $\operatorname{Li}(x) = \dfrac{x}{\ln x} - \dfrac{2}{\ln 2} + \displaystyle\int_2^x\frac{dt}{(\ln t)^2}$ by integration by parts. Then evaluate the partial sums $\dfrac{x}{\ln x}\left(1 + \dfrac{1}{\ln x} + \dfrac{2}{(\ln x)^2}\right)$ at $x = 10^8$ and compare with $\operatorname{Li}(10^8)$ and $\pi(10^8)$.
8. The prime number theorem implies that the $n$-th prime satisfies $p_n \sim n\ln n$. Find $p_{10^6}$ with the sieve and compare with $n\ln n$ and with the refinement $n(\ln n + \ln\ln n - 1)$.
9. Check that $\operatorname{Li}(n) > \pi(n)$ at every $n = 10^2, \dots, 10^9$, and compare the difference with $\sqrt n/\ln n$. (J. E. Littlewood proved in 1914 that $\pi(n) - \operatorname{Li}(n)$ nevertheless changes sign infinitely often; the first sign change is far beyond anything computed directly.)

## 7. Solutions

1. $\ln 10^6 \approx 6 \times 2.3 = 13.8$, so $\pi(10^6) \approx 10^6 / 13.8 \approx 72{,}500$ (with the exact logarithm, $72{,}382$). The true value is $78{,}498$, so the estimate is about $8\%$ low.
2. $1/\ln(500{,}000) = 1/13.12 \approx 0.0762$: about $1$ in $13$. The interval of $20{,}000$ numbers should then hold about $1{,}524$ primes; the sieve finds $1{,}525$.
3. $2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47$: fifteen primes. $50/\ln 50 \approx 12.8$ and $\operatorname{Li}(50) \approx 17.4$ — at this tiny scale neither is accurate, but they bracket the truth.
4. Near $10^{12}$: $\ln 10^{12} = 12 \ln 10 \approx 27.6$. Near $10^{100}$: $100\ln 10 \approx 230$.
5. $\pi(10^{100}) - \pi(10^{99}) \approx \dfrac{10^{100}}{230.26} - \dfrac{10^{99}}{227.96} \approx 4.343 \times 10^{97} - 0.439 \times 10^{97} \approx 3.9 \times 10^{97}$. There are $9 \times 10^{99}$ numbers with $100$ digits, so the fraction is about $3.9 \times 10^{97} / 9 \times 10^{99} \approx 1/231$, and among odd numbers about $1/115$.
6. At $10^7$: Legendre $\approx 665{,}140$ (error $+561$), $n/(\ln n - 1) \approx 661{,}459$ (error $-3{,}120$), $\operatorname{Li} \approx 664{,}917$ (error $+338$). At $10^9$: Legendre $\approx 50{,}917{,}519$ (error $+69{,}985$), $n/(\ln n - 1) \approx 50{,}701{,}542$ (error $-145{,}992$), $\operatorname{Li} \approx 50{,}849{,}234$ (error $+1{,}700$). Legendre's constant was fitted to the prime tables of his time and does very well up to about $10^7$, but it drifts away for larger $n$: the best constant $c$ in $n/(\ln n - c)$ slowly tends to $1$. $\operatorname{Li}$ is best at both.
7. With $u = 1/\ln t$, $dv = dt$: $du = -\dfrac{dt}{t(\ln t)^2}$, $v = t$, so $\displaystyle\int_2^x\frac{dt}{\ln t} = \Big[\frac{t}{\ln t}\Big]_2^x + \int_2^x\frac{dt}{(\ln t)^2}$. Repeating gives the series. At $10^8$ ($\ln x = 18.42$): $5{,}428{,}681 \times (1 + 0.0543 + 0.0059) \approx 5{,}755{,}384$, versus $\operatorname{Li}(10^8) \approx 5{,}762{,}208$ and $\pi(10^8) = 5{,}761{,}455$.
8. $p_{10^6} = 15{,}485{,}863$. $n\ln n = 13{,}815{,}511$ (about $11\%$ low); $n(\ln n + \ln\ln n - 1) \approx 15{,}441{,}302$ (about $0.3\%$ low).
9. Yes: $\operatorname{Li}(n) - \pi(n)$ is positive at every power of $10$ from $10^2$ to $10^9$ (from about $4$ at $10^2$ up to about $1{,}700$ at $10^9$). The ratio to $\sqrt n/\ln n$ stays between about $1.1$ and $1.9$ — of order $1$, so in this range the error is roughly the size of $\sqrt n/\ln n$, tiny compared with $\pi(n)$ itself.

Numerical checks:

In [ ]:
import math

import numpy as np

print("Ex 1:", round(10**6 / 13.8), round(10**6 / math.log(10**6)), " error:", round(10**6 / math.log(10**6) / 78498 - 1, 4))
print("Ex 2:", round(1 / math.log(500_000), 4), round(20_000 / math.log(500_000)),
      int(prime_count(510_000, primes) - prime_count(490_000, primes)))
print("Ex 3:", int(prime_count(50, primes)), round(50 / math.log(50), 2), round(Li(50), 2))
print("Ex 4:", round(12 * math.log(10), 2), round(100 * math.log(10), 2))
hundred_digit = 10**100 / (100 * math.log(10)) - 10**99 / (99 * math.log(10))
print("Ex 5:", f"{hundred_digit:.3e}", " fraction 1 in", round(9e99 / hundred_digit), " odd: 1 in", round(9e99 / hundred_digit / 2))
for n, true_count in [(10**7, 664_579), (10**9, 50_847_534)]:
    legendre = n / (math.log(n) - 1.08366)
    minus_one = n / (math.log(n) - 1)
    print(f"Ex 6: n = {n:.0e}: Legendre {legendre:,.0f} ({legendre - true_count:+,.0f}),  "
          f"ln n - 1: {minus_one:,.0f} ({minus_one - true_count:+,.0f}),  Li: {Li(n):,.0f} ({Li(n) - true_count:+,.0f})")
x = 10**8
log_x = math.log(x)
print("Ex 7:", round(x / log_x * (1 + 1 / log_x + 2 / log_x**2)), round(Li(x)), int(prime_count(x, primes)))
n = 10**6
print("Ex 8:", int(primes[n - 1]), round(n * math.log(n)), round(n * (math.log(n) + math.log(math.log(n)) - 1)))
for exponent in range(2, 10):
    n = 10**exponent
    true_count = pi_billion if exponent == 9 else int(prime_count(n, primes))
    difference = Li(n) - true_count
    print(f"Ex 9: 10^{exponent}: Li - pi = {difference:9.1f}   ratio to sqrt(n)/ln(n) = {difference / (math.sqrt(n) / math.log(n)):.3f}")

## 8. Conclusion

- $\pi(n)$, the number of primes up to $n$, grows like $n/\ln n$: the **prime number theorem**, conjectured from tables by Gauss and Legendre and proved in 1896 by Hadamard and de la Vallée Poussin.
- The cleanest way to think of it: **the density of primes near $n$ is $1/\ln n$**. Each factor of $10$ in $n$ adds the same $\ln 10 \approx 2.3$ to the typical gap — a multiplicative change in $n$ becomes an additive change in the spacing.
- Adding up the density gives $\operatorname{Li}(n) = \int_2^n dt/\ln t$, which is much more accurate than $n/\ln n$: at $10^9$ it is off by $1{,}700$ out of $50{,}847{,}534$, while $n/\ln n$ is off by $5\%$.
- The average gap between primes near $n$ is $\ln n$; individual gaps vary widely around it.
- Computationally: a NumPy sieve counts the primes below $10^9$ in seconds; $\operatorname{Li}$ is computed by substituting $t = e^u$ and integrating $e^u/u$ with the trapezoid rule.

This completes the **Logarithms in calculus** module: logarithms differentiate products and powers (01), appear as integrals (02), solve growth equations (03), estimate factorials (04) and count primes (05).

**Next:** module 05 opens with [Bits, digits and counting](../05_information_and_probability/01_bits_digits_and_counting.ipynb), where $\log_2$ and $\log_{10}$ measure how many questions or digits it takes to pin down a number.